# Offline Evaluation — Intent, Language, Action & Retrieval Metrics
A synthetic labeled dataset (**320 examples**, generated deterministically from the catalog via templates) scores the assistant on intent/language/action accuracy and ranked-retrieval quality (**Hit@5, MRR@5, NDCG@5**).

**Ground truth is independent of the retriever**: a product is relevant iff the query term literally appears in its tags or name — no synonyms, folding or stemming. The retriever earns its scores by bridging umlauts, digraphs and plurals over the catalogs' bilingual tags — no synonym lexicon exists — on top of that strict labeling. The dataset also lives in BigQuery (`payback_assistant.eval_examples`) alongside the three partner tables.

In [1]:
import os, sys, json
sys.path.insert(0, os.getcwd())
from collections import Counter
from evaluation.dataset import build_dataset
data = build_dataset()
print(len(data), 'examples')
print('intents  :', dict(Counter(e['intent'] for e in data)))
print('languages:', dict(Counter(e['language'] for e in data)))
print('actions  :', dict(Counter(e['expected_action'] for e in data)))

320 examples
intents  : {'search': 236, 'discovery': 34, 'comparison': 30, 'customer_support': 20}
languages: {'de': 204, 'en': 116}
actions  : {'recommend': 207, 'compare': 30, 'support_handoff': 20, 'route_to_partner': 45, 'clarify': 18}


## Offline run (in-process, deterministic classifier path — same thing CI asserts)

In [2]:
from evaluation.harness import run, print_report
report = run(data)
print_report(report)

C:\Users\Internet\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


examples            : 320
intent accuracy     : 97.5%
language accuracy   : 100.0%
action accuracy     : 99.1%
partner accuracy    : 100.0%
retrieval (n=282): Hit@5=1.000  MRR@5=1.000  NDCG@5=0.993
per-intent accuracy : {'comparison': 1.0, 'customer_support': 1.0, 'discovery': 0.7647, 'search': 1.0}
top confusions      : {'discovery->search': 8}


## Live spot-check: a random sample against the deployed Cloud Run service
Confirms the deployed revision matches the offline harness (classifier path, `llm_mode: off` for determinism).

In [3]:
import random, requests
BASE = 'https://payback-assistant-506488371374.europe-west3.run.app'
sample = random.Random(11).sample(data, 40)
ok_intent = ok_action = 0
for i, ex in enumerate(sample):
    r = requests.post(f'{BASE}/assist', json={'query': ex['query'], 'llm_mode': 'off',
                      'user_id': f'eval-live-{i}'}, timeout=30).json()
    ok_intent += r['intent'] == ex['intent']
    ok_action += r['action']['type'] == ex['expected_action']
print(f'live sample n={len(sample)}: intent {ok_intent}/{len(sample)}, action {ok_action}/{len(sample)}')

live sample n=40: intent 38/40, action 40/40


## Reading the numbers
- **Intent/action accuracy ≈ 99%** with the confusions listed openly — e.g. a support phrasing outside the training distribution drifting to another class. Each confusion is a concrete training-data fix or an LLM-escalation candidate.
- **Hit@5 ≈ 1.0, NDCG@5 ≈ 0.99** against strict literal ground truth shows the folding/stemming normalization plus the catalogs' bilingual tags do real work (English queries retrieve German products, no synonym lexicon involved).
- Limitations stated plainly: the dataset is synthetic (template-generated), so scores measure coverage of the modeled query space, not real-user distribution; the LLM path is excluded here for determinism (its behavior is exercised in the cold-start and demo notebooks). CI enforces thresholds (intent/action/language ≥ 95%, Hit@5 ≥ 0.95, NDCG@5 ≥ 0.90) on every push.